# YOLO系列

## YOLOv1

1.  系统将图像划分为 $S \times S$ 网格（如 $7 \times 7$）。
2.  每个网格预测 $B$ 个边界框 $(x,y,w,h,\text{confidence})$ 与 $C$ 个类别的条件概率 $\text{Pr}(\text{Class}_i|\text{Object})$。
3.  边界框参数定义：$(x,y)$ 为框中心相对网格边界的位置；$(w,h)$ 为宽高相对于整张图的比例；$\text{confidence} = \text{Pr}(\text{Object}) \times \text{IOU}_{\text{pred}}^{\text{truth}}$。
4.  测试时将类别条件概率与置信度相乘，得到 $\text{Pr}(\text{Class}_i) \times \text{IOU}_{\text{pred}}^{\text{truth}}$，同时编码类别概率与框的匹配度。
5.  最终预测结果编码为 $S \times S \times (B \times 5 + C)$ 张量。
6.  在 $\text{PASCAL VOC}$ 上评估 YOLO 时，取 $S=7, B=2, C=20$，输出张量尺寸为 $7 \times 7 \times 30 = 1470$。

__损失函数__
<div style="background-color:#f9f9f9; padding:10px; margin:auto; width: 80%">
    <img src="./1506_YOLO/assets/loss.png" />
</div>

$$\begin{aligned}
&\lambda_{\text{coord}} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{\text{obj}} \left[ (x_i - \hat{x}_i)^2 + (y_i - \hat{y}_i)^2 \right] \\
+&\lambda_{\text{coord}} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{\text{obj}} \left[ (\sqrt{w_i} - \sqrt{\hat{w}_i})^2 + (\sqrt{h_i} - \sqrt{\hat{h}_i})^2 \right]\\
+&\sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{\text{obj}} (C_i - \hat{C}_i)^2 \\
+&\lambda_{\text{noobj}} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}_{ij}^{\text{noobj}} (C_i - \hat{C}_i)^2 \\
+&\sum_{i=0}^{S^2}  \mathbb{1}_{i}^{\text{obj}} \sum_{c\in \text{classes}} (p_i(c) - \hat{p}_i(c))^2 & 
\end{aligned}$$

- $\sum_{i=0}^{S^2}$: $i$ 遍历所有网格单元。
- $\sum_{j=0}^{B}$: $j$ 遍历每个网格单元的边界框。
- $\mathbb{1}_{i}^{\text{obj}}$: 表示物体是否出现在第$i$个网格内。
- $\mathbb{1}_{ij}^{\text{obj}}$: 表示第$i$个网格的第$j$个边界框负责预测该物体。 ($^{\text{noobj}}$表示没有物体)
  - 实际操作中，加入第$i$个网格有$j$个预测框，我们使用与真实边界框 $ \text{IOU}_{\text{pred}}^{\text{truth}}$ 最大的那个预测框来负责预测该物体。
  - 其他预测框的$\mathbb{1}_{ij}^{\text{obj}}=0$




- 第一部分: 坐标预测损失 = (预测框中心坐标误差 + 预测框宽高误差)，使用$\lambda_{\text{coord}}$加权
  - $(x_i, y_i, w_i, h_i)$: 第$i$个网格单元的第$j$个边界框的真实坐标。(加 $\hat{}$ 表示预测值)
  - 中心坐标误差使用平方误差
  - 宽高误差使用平方根，平衡框大小对损失的影响
- 第二部分: 含有obj的预测框的置信度损失 = 预测框置信度误差
  - $C_i$: 第$i$个网格单元的第$j$个边界框的真实置信度。（人为标记，取值1） (加 $\hat{}$ 表示预测值)
  - 目标：让$\hat{C}_i \rightarrow 1$
- 第三部分: 不含obj的预测框的置信度损失 = 预测框置信度误差，使用$\lambda_{\text{noobj}}$加权
  - $C_i$: 第$i$个网格单元的第$j$个边界框的真实置信度。（人为标记，取值0） (加 $\hat{}$ 表示预测值)
  - 目标：让$\hat{C}_i \rightarrow 0$
- 第四部分: 类别条件概率损失 = 预测类别条件概率误差
  - $p_i(c)$: 第$i$个网格单元的类别$c$的真实条件概率（出现为1，未出现为0）。 (加 $\hat{}$ 表示预测值)

## YOLOv2

```txt
 layer    type   filters     size / stride             input            output
------------------------------------------------------------------------------
          conv        32    3 x 3 / 1       416 x 416 x    3  416 x 416 x   32
     1     max              2 x 2 / 2       416 x 416 x   32  208 x 208 x   32
     2    conv        64    3 x 3 / 1       208 x 208 x   32  208 x 208 x   64
     3     max              2 x 2 / 2       208 x 208 x   64  104 x 104 x   64
     4    conv       128    3 x 3 / 1       104 x 104 x   64  104 x 104 x  128
     5    conv        64    1 x 1 / 1       104 x 104 x  128  104 x 104 x   64
     6    conv       128    3 x 3 / 1       104 x 104 x   64  104 x 104 x  128
     7     max              2 x 2 / 2       104 x 104 x  128   52 x  52 x  128
     8    conv       256    3 x 3 / 1        52 x  52 x  128   52 x  52 x  256
     9    conv       128    1 x 1 / 1        52 x  52 x  256   52 x  52 x  128
    10    conv       256    3 x 3 / 1        52 x  52 x  128   52 x  52 x  256
    11     max              2 x 2 / 2        52 x  52 x  256   26 x  26 x  256
    12    conv       512    3 x 3 / 1        26 x  26 x  256   26 x  26 x  512
    13    conv       256    1 x 1 / 1        26 x  26 x  512   26 x  26 x  256
    14    conv       512    3 x 3 / 1        26 x  26 x  256   26 x  26 x  512
    15    conv       256    1 x 1 / 1        26 x  26 x  512   26 x  26 x  256
    16    conv       512    3 x 3 / 1        26 x  26 x  256   26 x  26 x  512
    17     max              2 x 2 / 2        26 x  26 x  512   13 x  13 x  512
    18    conv      1024    3 x 3 / 1        13 x  13 x  512   13 x  13 x 1024
    19    conv       512    1 x 1 / 1        13 x  13 x 1024   13 x  13 x  512
    20    conv      1024    3 x 3 / 1        13 x  13 x  512   13 x  13 x 1024
    21    conv       512    1 x 1 / 1        13 x  13 x 1024   13 x  13 x  512
    22    conv      1024    3 x 3 / 1        13 x  13 x  512   13 x  13 x 1024
    23    conv      1024    3 x 3 / 1        13 x  13 x 1024   13 x  13 x 1024
    24    conv      1024    3 x 3 / 1        13 x  13 x 1024   13 x  13 x 1024
    25   route        16                                                      
    26    conv        64    1 x 1 / 1        26 x  26 x  512   26 x  26 x   64
    27   reorg                    / 2        26 x  26 x   64   13 x  13 x  256
    28   route     27 24                                                      
    29    conv      1024    3 x 3 / 1        13 x  13 x 1280   13 x  13 x 1024
    30    conv       125    1 x 1 / 1        13 x  13 x 1024   13 x  13 x  125
```

1. 骨干网络：使用更深的网络（Darknet-19），全卷积，使用BN层
2. 特征融合：Passthrough层简单融合
   - 将浅层高分辨率与深层低分辨率特征拼接：
     - Conv: $26\times26\times512 \rightarrow 26\times26\times64$
     - Split + Concat: $26\times26\times64 \rightarrow 13\times13\times256$
     - 叠加: $13\times13\times256, 13\times13\times1024 \rightarrow 13\times13\times1280$
3. k-means聚类5个锚框
   - 提取所有GT的宽高$wh$
   - 普通K-means用欧式距离($\sqrt{(w_i - w_j)^2 + (h_i - h_j)^2}$)，不适合边界框聚类
   - 这里用$d = 1 - IOU(\text{box}, \text{centroid})$作为距离度量
   - 随机选择5个框开始，遍历、分配、重新计算
   - 产出$p_w, p_h$, 
4. 检测尺度：单尺度(448x448) + 多尺度训练
5. 直接位置预测(Direct Location Prediction): 相比于yolov1的坐标回归，更稳定
   - RPN: 人工预设$w_\alpha, h_\alpha$
     - $x = (\textcolor{green}{t_x} \cdot w_\alpha) - x_a$
     - $y = (\textcolor{green}{t_y} \cdot h_\alpha) - y_a$
     - $x_a, y_a$: 锚框原始中心坐标
     - $w_\alpha, h_\alpha$: 人工预设的锚框宽高
    - YOLOv2中的算法：  $$\begin{aligned}
          b_x &= \sigma(\textcolor{green}{t_x}) + c_x \\
          b_y &= \sigma(\textcolor{green}{t_y})  + c_y\\
          b_w &= p_w e^{\textcolor{green}{t_w}}\\
          b_h &= p_h e^{\textcolor{green}{t_h}}\\
          \Pr(\text{object}) &\cdot \text{IOU}(b, \text{object}) = \sigma(\textcolor{green}{t_o})\\
          \end{aligned}$$  
       - $b_x, b_y$: 预测框中心坐标
         - 格子左上角坐标$c_x, c_y$(多少个) + 偏移量$\sigma(\textcolor{green}{t_x}), \sigma(\textcolor{green}{t_y})$
         - 因为格子大小被归一化为1，$\sigma$是sigmoid函数，结果在0-1之间，所以最终的预测框中心在格子内
       - $p_w, p_h$: 原始锚框宽高
       - $b_w, b_h$: 预测框宽高
       - $\textcolor{green}{t_x}, \textcolor{green}{t_y}, \textcolor{green}{t_w}, \textcolor{green}{t_h}, \textcolor{green}{t_o}$: 网络预测值
6. 分类：Softmax(logits) -> 1000个类别概率
7. 小目标检测：For VOC: $B=5, C=20$, 5个锚框 * (5个参数 + 20类别概率) = 125
8. 精读/速度：召回率提升，速度提升

## YOLOv3

__模型结构:__

<!-- <div style="background-color:white; padding:10px; margin:auto; width: 30%">
    <img src="./1804_YOLOv3/assets/arch2.png" />
</div> -->

<div style="background-color:white; padding:10px; margin:auto; width: 80%">
    <img src="./assets/yolov3_head.png" />
    <span style="text-align:center; display:block; color:black">YOLOv3检测模型：检测头输出3个尺度的特征图（FPN多尺度特征融合）</span>
</div>

- YOLOv3改进(相对于YOLOv2)：
  - 骨干网络：使用更深的网络（Darknet-53），使用残差连接
  - 特征融合：FPN多尺度特征融合
    - 每个尺度预测三个框：$N × N × [3 ∗ (4 + 1 + 80)]$
  - k-means在COCO数据集上聚类了9个锚框（每个尺度3个）
  - 检测尺度：三尺度(320x320, 416x416, 608x608) + 多尺度训练
  - 分类：独立的Logistic回归 多标签
  - 小目标检测：FPN多尺度检测
  - 精读/速度：mAP提升，速度略有下降
  - 残差连接，作用：
    1. 稳梯度传播：反向时梯度可沿残差捷径直达浅层，避免深层链式求导导致的梯度消失
    2. 防网络退化：加深时精度不下降，恒等映射保底 “至少不更差”，让深层网络更易收敛。
    3. 强特征复用：低层细节（边缘、纹理）与高层语义同时传递，提升小目标定位与分类精度。

## YOLOv4

- 骨干网络Backbone：使用了CSPDarknet53代替了之前的Darknet53网络。融合了CSPNet技术。
  - CSP：将特征图按通道维度一分为2，一部分不做处理，一部分通过卷积块，最后再拼接
  - 用CSP代替了残差连接，减少计算量，提升速度
- 路径聚合网络PANet：用于特征融合，将不同尺度的特征进行聚合，以便在多尺度上检测大小不同的目标。
- 预测头Head：使用YOLOv3风格的检测头，在不同尺寸上进行目标检测。

__模型结构:__  

<div style="background-color:#f9f9f9; padding:10px; margin:auto; width: 80%">
    <img src="./2004_YOLOv4/assets/detector.png" />
</div>

__修改后的PAN和SAM模块:__

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin: auto; display: grid; 
    grid-template-columns: 1fr 1fr; /* 两列均等 */ 
    grid-template-rows: auto auto; /* 第一行图片，第二行标题 */ 
    gap: 10px 20px; /* 行间距 10px，列间距 20px */ 
align-items: center; justify-items: center;">
    <img src="./2004_YOLOv4/assets/pan.png" style="max-width: 100%; height: auto;" />
    <img src="./2004_YOLOv4/assets/sam.png" style="max-width: 100%; height: auto;" />
    <div style="font-size: 12px; color: black; width: 100%; text-align: center;">
        <strong>图(a)</strong>：修改后的PAN模块
    </div>
    <div style="font-size: 12px; color: black; width: 100%; text-align: center;">
        <strong>图(b)</strong>：修改后的SAM模块
    </div>
</div>

#### CSPNet
- [论文](https://arxiv.org/abs/1911.11929)  

<div style="background-color:#f9f9f9; padding:10px; margin:auto; width: 80%">
    <img src="./assets/CSPNet.png" />
</div>

- DenseNet中每一部分都与当前部分的卷积结果拼接，这样会增加计算量。
- CSPNet则将特征图分成两部分，一部分经过残差连接，另一部分直接与输出拼接，减少计算量同时保持梯度流动。

## YOLOv5

- 使用`SPPF`代替了`SPP`模块，提升速度
  - 串行小核池化代替并行大核池化，速度提升约30%

```python
# SPP与SPPF对比（示例代码，并非官方实现）
class SPP(nn.Module):
    def __init__(self):
        super().__init__()
        self.maxpool1 = nn.MaxPool2d(kernel_size = 5, stride = 1, padding=2)
        self.maxpool2 = nn.MaxPool2d(9, 1, padding=4)
        self.maxpool3 = nn.MaxPool2d(13, 1, padding=6)
    def forward(self, x):
        o1 = self.maxpool1(x)
        o2 = self.maxpool2(x)
        o3 = self.maxpool3(x)
        return torch.cat([x, o1, o2, o3], dim=1)
class SPPF(nn.Module):
    def __init__(self):
        super().__init__()
        self.maxpool = nn.MaxPool2d(5, 1, padding=2)
    def forward(self, x):
        o1 = self.maxpool(x)
        o2 = self.maxpool(o1)
        o3 = self.maxpool(o2)
        return torch.cat([x, o1, o2, o3], dim=1)
```

In [ ]:
import torch
import torch.nn as nn
from models.common import Conv, Bottleneck

class C3(nn.Module):
    """Implements a CSP Bottleneck module with three convolutions for enhanced feature extraction in neural networks."""

    def __init__(self, c1, c2, n=1, shortcut=True, g=1, e=0.5):
        """Initializes C3 module with options for channel count, bottleneck repetition, shortcut usage, group
        convolutions, and expansion.
        """
        super().__init__()
        c_ = int(c2 * e)  # hidden channels
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(c1, c_, 1, 1)
        self.cv3 = Conv(2 * c_, c2, 1)  # optional act=FReLU(c2)
        self.m = nn.Sequential(*(Bottleneck(c_, c_, shortcut, g, e=1.0) for _ in range(n)))

    def forward(self, x):
        """Performs forward propagation using concatenated outputs from two convolutions and a Bottleneck sequence."""
        return self.cv3(torch.cat((self.m(self.cv1(x)), self.cv2(x)), 1))  # cv3([yn, a])

## YOLOv8

- 使用`C2f`模块代替了YOLOv5中的`C3`模块：
  - `C3`模块：分两部分，一部分通过卷积块，一部分通过一系列瓶颈块，最后拼接再卷积。
  - `C2f`：分两部分，一部分通过卷积块，一部分通过一系列瓶颈块，但是每次都保留输入的特征图，最后拼接再卷积。

In [ ]:
import torch
import torch.nn as nn
from ultralytics.nn.modules import Conv, Bottleneck
class C2f(nn.Module):
    """Faster Implementation of CSP Bottleneck with 2 convolutions."""

    def __init__(self, c1: int, c2: int, n: int = 1, shortcut: bool = False, g: int = 1, e: float = 0.5):
        """Initialize a CSP bottleneck with 2 convolutions.

        Args:
            c1 (int): 输入通道数
            c2 (int): 输出通道数
            n (int): Bottleneck块的数量。
            shortcut (bool): 是否使用快捷连接。
            g (int): 卷积的组数。
            e (float): 扩展比例。
        """
        super().__init__()
        self.c = int(c2 * e)  # 隐藏通道数，默认为输出通道的50%
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1)  # optional act=FReLU(c2)
        self.m = nn.ModuleList(Bottleneck(self.c, self.c, shortcut, g, k=((3, 3), (3, 3)), e=1.0) for _ in range(n))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """C2f的前向传播"""
        y = list(self.cv1(x).chunk(2, 1))  # cv1的输出在第1维度(通道)切分为两部分
        y.extend(m(y[-1]) for m in self.m)  # 对最后一部分进行n个Bottleneck块的处理
        # HL: y = [a, b, y1, ..., yn], 在C3的基础上多了b, y1, ..., yn-1 这n个中间结果
        return self.cv2(torch.cat(y, 1))  # 在第1个维度上拼接所有部分后通过cv2

    def forward_split(self, x: torch.Tensor) -> torch.Tensor:
        """使用split()代替chunk()的前向传播"""
        y = self.cv1(x).split((self.c, self.c), 1)
        y = [y[0], y[1]]
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))